In [1]:
# Import required libraries
import json
import random
import math
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Optional, Tuple
from copy import deepcopy
import matplotlib.pyplot as plt
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Import your ShapeSceneGenerator class
from render_svg_mutator import ShapeSceneGenerator

# Import the VLMExperimentFramework class code
from perturbation_vlm_experiment_framework import VLMExperimentFramework


VLM Experiment Framework - Usage Instructions

1. Run a single experiment:
   framework, experiment, vlm_test, pairwise, mc = run_single_experiment_demo()

2. Run batch experiments:
   framework, batch, summary = run_batch_experiments_demo()

3. Create custom experiment:
   framework = VLMExperimentFramework()
   exp = framework.create_single_experiment(
       experiment_name="my_experiment",
       scene_type="random",
       num_mutations=5,
       intensity_range=(0.1, 0.9)
   )

4. Visualize experiment:
   framework.visualize_experiment(exp)

5. Generate VLM test data:
   vlm_test = framework.prepare_vlm_test_set(exp)
   pairwise = framework.create_pairwise_comparisons(exp)
   mc = framework.create_multiple_choice_test(exp)



In [2]:
# Initialize the experiment framework
framework = VLMExperimentFramework(
    output_dir="perturbation_vlm_experiments",
    canvas_size=600
)

print("VLM Experiment Framework initialized!")
print(f"Output directory: {framework.output_dir}")

VLM Experiment Framework initialized!
Output directory: perturbation_vlm_experiments


### Batch Experiment Test

In [3]:
#===================================
#= create the images test dataset ==
#===================================


batch_experiments = framework.create_batch_experiments(
    num_experiments=50,
    scene_type="random",
    batch_name="batch_02_flash",
    fixed_ranges=True
)

# save the batch experiments metadata
for exp in batch_experiments:
    exp_dir = Path(exp["experiment_dir"])
    with open(exp_dir / f"{exp['experiment_name']}_metadata.json", 'w') as f:
        json.dump(exp, f, indent=2)

Created batch_02_flash_001: random scene, intensities [0.06, 0.24, 0.55, 0.62, 0.88]
Created batch_02_flash_002: random scene, intensities [0.08, 0.33, 0.47, 0.78, 0.89]
Created batch_02_flash_003: random scene, intensities [0.11, 0.29, 0.41, 0.67, 0.99]
Created batch_02_flash_004: random scene, intensities [0.1, 0.39, 0.46, 0.68, 0.98]
Created batch_02_flash_005: random scene, intensities [0.05, 0.24, 0.4, 0.71, 0.85]
Created batch_02_flash_006: random scene, intensities [0.09, 0.21, 0.5, 0.72, 0.9]
Created batch_02_flash_007: random scene, intensities [0.18, 0.35, 0.57, 0.78, 0.96]
Created batch_02_flash_008: random scene, intensities [0.05, 0.23, 0.48, 0.62, 0.81]
Created batch_02_flash_009: random scene, intensities [0.03, 0.35, 0.41, 0.62, 0.89]
Created batch_02_flash_010: random scene, intensities [0.07, 0.31, 0.48, 0.65, 0.97]
Created batch_02_flash_011: random scene, intensities [0.01, 0.37, 0.5, 0.71, 0.93]
Created batch_02_flash_012: random scene, intensities [0.03, 0.21, 0.4

In [4]:
# Comprehensive Analysis of Batch VLM Experiments with Per-Experiment Logging
# This analyzes all your experiment folders and saves logs in each folder

import os
import sys
import logging
import json
from pathlib import Path
from datetime import datetime

print("="*80)
print("ANALYZING BATCH VLM EXPERIMENTS")
print("="*80)

# Configuration
EXPERIMENTS_DIR = "perturbation_vlm_experiments"
MODEL_NAME = "gemini-2.5-flash"  # Change to your working model
from agent.agent_svg import Agent
from render_svg import SVGAgent

def setup_experiment_logger(experiment_folder: Path, experiment_name: str):
    """Setup a logger that saves to the specific experiment folder"""
    
    # Create a unique logger for this experiment
    logger_name = f"experiment_{experiment_name}"
    logger = logging.getLogger(logger_name)
    
    # Clear any existing handlers
    logger.handlers = []
    
    # Set logging level
    logger.setLevel(logging.INFO)
    
    # Create log file in the experiment folder
    log_file = experiment_folder / f"{experiment_name}_analysis.log"
    
    # Create file handler
    file_handler = logging.FileHandler(log_file, mode='w')  # 'w' to overwrite if exists
    file_handler.setLevel(logging.INFO)
    
    # Create formatter
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    file_handler.setFormatter(formatter)
    
    # Add handler to logger
    logger.addHandler(file_handler)
    
    return logger

def save_experiment_results(experiment_folder: Path, experiment_name: str, results: dict):
    """Save detailed experiment results to JSON file in experiment folder"""
    
    results_file = experiment_folder / f"{experiment_name}_results.json"
    
    # Add timestamp and metadata
    results_with_metadata = {
        "experiment_name": experiment_name,
        "analysis_timestamp": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        **results
    }
    
    with open(results_file, 'w') as f:
        json.dump(results_with_metadata, f, indent=2)
    
    return results_file

batch_experiments_results = {}
# Check if experiments directory exists
if not Path(EXPERIMENTS_DIR).exists():
    print(f"❌ Experiments directory not found: {EXPERIMENTS_DIR}")
    print("Please check the path and try again.")
else:
    # Count experiment folders
    experiment_folders = [d for d in Path(EXPERIMENTS_DIR).iterdir() 
                         if d.is_dir() and d.name.startswith("batch_02_flash")]
    experiment_folders.sort(key=lambda x: x.name)  # Sort by folder name
    print(f"Found {len(experiment_folders)} experiment folders")
    
    if len(experiment_folders) == 0:
        print("No experiment folders found. Make sure they start with 'batch_'")
    else:
        print("Starting analysis with per-experiment logging...")
        
        for i, folder in enumerate(experiment_folders, 1):
            experiment_name = folder.name
            print(f"\n📁 Processing {i}/{len(experiment_folders)}: {experiment_name}")
            
            # Setup logger for this specific experiment
            exp_logger = setup_experiment_logger(folder, experiment_name)
            exp_logger.info(f"Starting analysis of experiment: {experiment_name}")
            exp_logger.info(f"Model used: {MODEL_NAME}")
            
            try:
                # Load original and candidate images
                gt_images = [str(image) for image in folder.glob("*") if 'original' in image.stem]
                gt_image = gt_images[0] if gt_images else None
                
                if gt_image is None:
                    error_msg = f"No ground truth image found in {folder}"
                    exp_logger.error(error_msg)
                    print(f"❌ {error_msg}")
                    continue
                
                exp_logger.info(f"Ground truth image: {Path(gt_image).name}")
                print(f"   📸 GT: {Path(gt_image).name}")
                
                candidate_images = [str(image) for image in folder.glob("*") if 'intensity' in image.stem]
                
                if not candidate_images:
                    error_msg = f"No candidate images found in {folder}"
                    exp_logger.error(error_msg)
                    print(f"❌ {error_msg}")
                    continue
                
                exp_logger.info(f"Found {len(candidate_images)} candidate images")
                exp_logger.info(f"Candidate images: {[Path(img).name for img in candidate_images]}")
                print(f"   🖼️  Candidates: {len(candidate_images)} images")
                
                # Extract the float in candidate image paths stem and assign the path with the smallest one as the reference image
                reference_image = min(candidate_images, key=lambda x: float(Path(x).stem.split('_')[-1]))
                reference_intensity = float(Path(reference_image).stem.split('_')[-1])
                
                exp_logger.info(f"Reference image (lowest intensity): {Path(reference_image).name}")
                exp_logger.info(f"Reference intensity: {reference_intensity}")
                print(f"   🎯 Reference: {Path(reference_image).name} (intensity: {reference_intensity})")
                
                # Instantiate and apply the VLM agent
                exp_logger.info(f"Initializing VLM agent with model: {MODEL_NAME}")
                agent = Agent(model_name=MODEL_NAME, target_image_path=gt_image)
                agent_png = SVGAgent(600, 600)
                
                exp_logger.info("Starting VLM evaluation...")
                vlm_start_time = datetime.now()
                
                vlm_response = agent.vlm_judge_best_candidate(
                    target_image_path=gt_image, 
                    candidate_paths=candidate_images
                )
                
                vlm_end_time = datetime.now()
                vlm_duration = (vlm_end_time - vlm_start_time).total_seconds()
                
                exp_logger.info(f"VLM evaluation completed in {vlm_duration:.2f} seconds")
                
                # Extract VLM results
                best_image = vlm_response.get("best_candidate")
                vlm_confidence = vlm_response.get("confidence", 0.0)
                vlm_scores = vlm_response.get("all_scores", [])
                vlm_analysis = vlm_response.get("analysis", "")
                
                exp_logger.info(f"VLM chosen best candidate: {Path(best_image).name if best_image else 'None'}")
                exp_logger.info(f"VLM confidence: {vlm_confidence}")
                exp_logger.info(f"VLM scores: {vlm_scores}")
                
                # Compare results
                success = best_image == reference_image
                
                print(f"   🤖 VLM chose: {Path(best_image).name if best_image else 'None'}")
                print(f"   📊 Confidence: {vlm_confidence:.3f}")
                
                if success:
                    exp_logger.info("✅ SUCCESS: VLM choice matches reference image")
                    print(f"   ✅ Success!")
                    batch_experiments_results["success"] = batch_experiments_results.get("success", 0) + 1
                else:
                    exp_logger.warning("❌ FAILURE: VLM choice does not match reference image")
                    exp_logger.warning(f"Expected: {Path(reference_image).name}")
                    exp_logger.warning(f"Got: {Path(best_image).name if best_image else 'None'}")
                    print(f"   ❌ Failure!")
                    print(f"      Expected: {Path(reference_image).name}")
                    print(f"      Got: {Path(best_image).name if best_image else 'None'}")
                    batch_experiments_results["failure"] = batch_experiments_results.get("failure", 0) + 1
                
                # Save detailed results to the experiment folder
                experiment_results = {
                    "success": success,
                    "gt_image": gt_image,
                    "candidate_images": candidate_images,
                    "reference_image": reference_image,
                    "reference_intensity": reference_intensity,
                    "vlm_chosen_image": best_image,
                    "vlm_confidence": vlm_confidence,
                    "vlm_scores": vlm_scores,
                    "vlm_analysis": vlm_analysis,
                    "vlm_full_response": vlm_response,
                    "vlm_duration_seconds": vlm_duration,
                    "analysis_start_time": vlm_start_time.isoformat(),
                    "analysis_end_time": vlm_end_time.isoformat()
                }
                
                results_file = save_experiment_results(folder, experiment_name, experiment_results)
                exp_logger.info(f"Results saved to: {results_file.name}")
                
                exp_logger.info(f"Experiment {experiment_name} completed successfully")
                
            except Exception as e:
                error_msg = f"Error processing experiment {experiment_name}: {str(e)}"
                exp_logger.error(error_msg)
                exp_logger.exception("Full exception details:")
                print(f"   ❌ Error: {str(e)}")
                
                # Save error results
                error_results = {
                    "success": False,
                    "error": str(e),
                    "error_type": type(e).__name__
                }
                
                try:
                    save_experiment_results(folder, experiment_name, error_results)
                    exp_logger.info(f"Error results saved to: {experiment_name}_results.json")
                except Exception as save_error:
                    exp_logger.error(f"Failed to save error results: {save_error}")
                
                batch_experiments_results["failure"] = batch_experiments_results.get("failure", 0) + 1
            
            finally:
                # Close the logger handlers to free up file handles
                for handler in exp_logger.handlers:
                    handler.close()
                exp_logger.handlers = []

# Summarise the results
print("\n" + "="*80)
print("BATCH VLM EXPERIMENTS SUMMARY")
print("="*80)
total_experiments = len(experiment_folders)
success_count = batch_experiments_results.get('success', 0)
failure_count = batch_experiments_results.get('failure', 0)
success_rate = (success_count / total_experiments * 100) if total_experiments > 0 else 0

print(f"Total experiments: {total_experiments}")
print(f"Success: {success_count}")
print(f"Failure: {failure_count}")
print(f"Success Rate: {success_rate:.2f}%")

# Save overall summary
summary_file = Path(EXPERIMENTS_DIR) / f"batch_analysis_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
overall_summary = {
    "total_experiments": total_experiments,
    "success_count": success_count,
    "failure_count": failure_count,
    "success_rate": success_rate / 100,
    "model_name": MODEL_NAME,
    "analysis_timestamp": datetime.now().isoformat(),
    "experiments_directory": EXPERIMENTS_DIR
}

with open(summary_file, 'w') as f:
    json.dump(overall_summary, f, indent=2)

print(f"\n📄 Overall summary saved to: {summary_file}")
print(f"📁 Individual experiment logs and results saved in each experiment folder")

print("\n" + "="*80)
print("FILES CREATED PER EXPERIMENT:")
print("="*80)
print("In each experiment folder, you'll find:")
print("  📄 {experiment_name}_analysis.log - Detailed analysis log")
print("  📊 {experiment_name}_results.json - Complete results and VLM response")
print(f"  🖼️  Original images and candidates (already existed)")
print("\nOverall summary:")
print(f"  📋 {summary_file.name} - Batch summary statistics")


ANALYZING BATCH VLM EXPERIMENTS
Found 50 experiment folders
Starting analysis with per-experiment logging...

📁 Processing 1/50: batch_02_flash_001
   📸 GT: batch_02_flash_001_original.png
   🖼️  Candidates: 5 images
   🎯 Reference: batch_02_flash_001_mutation_01_intensity_0.06.png (intensity: 0.06)
   🤖 VLM chose: batch_02_flash_001_mutation_01_intensity_0.06.png
   📊 Confidence: 0.900
   ✅ Success!

📁 Processing 2/50: batch_02_flash_002
   📸 GT: batch_02_flash_002_original.png
   🖼️  Candidates: 5 images
   🎯 Reference: batch_02_flash_002_mutation_01_intensity_0.08.png (intensity: 0.08)
   🤖 VLM chose: batch_02_flash_002_mutation_01_intensity_0.08.png
   📊 Confidence: 0.900
   ✅ Success!

📁 Processing 3/50: batch_02_flash_003
   📸 GT: batch_02_flash_003_original.png
   🖼️  Candidates: 5 images
   🎯 Reference: batch_02_flash_003_mutation_01_intensity_0.11.png (intensity: 0.11)
   🤖 VLM chose: batch_02_flash_003_mutation_01_intensity_0.11.png
   📊 Confidence: 0.900
   ✅ Success!

📁 Pro

### single experiment demo

In [ ]:
# Run a single experiment with random scene and random intensities
from datetime import datetime
# read the current time as the experiment_name
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment = framework.create_single_experiment(
    experiment_name=current_time,
    scene_type="random",  # or "structured", "pattern"
    num_mutations=5,
    intensity_range=(0.1, 0.9),
    add_noise=True
)

print("Single Experiment Results:")
print(f"Experiment ID: {experiment['experiment_id']}")
print(f"Scene type: {experiment['scene_type']}")
print(f"Base scene shapes: {len(experiment['base_scene'])}")
print(f"Random intensities: {experiment['intensities']}")
print(f"Image paths:")
for label, intensity, path in experiment['image_paths']:
    print(f"  {label}: {intensity:.2f} -> {path}")

In [ ]:
# read all the images in the experiment path
path = Path(framework.output_dir) / current_time
gt_images = [str(image) for image in path.glob("*") if 'original' in image.stem]
gt_image = gt_images[0] if gt_images else None
candidate_images = [str(image) for image in path.glob("*") if 'intensity' in image.stem]

print(f"Ground-truth image: {gt_image}")
print(f"Candidate images: {candidate_images}")

# import the VLM agent to do image judgement
from agent.agent_svg import Agent
# import the svg render
from render_svg import SVGAgent

model_names = ['gemini-2.5-flash','gemini-2.5-pro']
# instantiate the agent
agent = Agent(model_name=model_names[0],target_image_path=gt_image)
# instantiate the SVG renderer
agent_png = SVGAgent(600,600)

# select the best image among 5 candidates
best_image = agent.vlm_judge_best_candidate(target_image_path=gt_image, candidate_paths=candidate_images) # path is where the original and candidate images are saved
print(f"Best candidate image: {best_image}")